## Testing Activation Functions with MNIST

This notebook tests the impact of different activation functions on test accuracy when training a feed forward network with the MNIST dataset. We will also time the overall execution of training through 5 epochs.

In [ ]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
from timer import Timer

In [ ]:
 t = Timer()

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Define Plotting Function

A useful plotting functions is defined here and will be used later to illustrate the shape of each activation function.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def arrowed_axes(ax):
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.spines['left'].set_position('zero')
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_position('zero')
    ax.spines['top'].set_visible(False)
    ax.axhline(y=0, color='k')
    ax.axvline(x=0, color='k')

    # make arrows
    ax.plot((1), (0), ls="", marker=">", ms=10, color="k",
            transform=ax.get_yaxis_transform(), clip_on=False)
    ax.plot((0), (1), ls="", marker="^", ms=10, color="k",
            transform=ax.get_xaxis_transform(), clip_on=False)

### Download MNIST

MNIST is built into PyTorch and so the data is loaded using <code>datasets.MNIST</code>.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
test_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

### Prepare Images

As in the first example with MNIST, the images are flattened to vectors of length 784 with the greyscale pixel intensity values converted to <code>float32</code> data type and scaled to lie between 0 and 1. Labels are converted to "one-hot" representation.

In [ ]:
train_x = train_set.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_set.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255
num_classes = train_set.targets.max() + 1
train_y = F.one_hot(train_set.targets, num_classes=num_classes).float()
test_y = F.one_hot(test_set.targets, num_classes=num_classes).float()

### Build Model

Swish and SERLU are not available in PyTorch 1.13 so we define them here.

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
def serlu(x):
    return 1.10786 * F.relu(x) + 3.1326 * torch.min(x, torch.tensor(0.0)) * torch.exp(torch.min(x, torch.tensor(0.0)))

class SERLU(nn.Module):
    def __init__(self):
        super(SERLU, self).__init__()
        
    def forward(self, x):
        return serlu(x)

A dictionary of activation functions is constructed so we can pass the name to the network

In [ ]:
activation_functions = {
    'relu': nn.ReLU(),
    'sigmoid': nn.Sigmoid(),
    'tanh': nn.Tanh(),
    'leaky_relu': nn.LeakyReLU(),
    'elu': nn.ELU(),
    'prelu' : nn.PReLU(),
    'selu' : nn.SELU(),
    'gelu' : nn.GELU(),
    'swish' : Swish(),
    'serlu' : SERLU()
}

We will use the same network architecture as the previous example with two hidden layers of 200 neurons and a softmax output layer with 10 neurons. However, the activation function will be varied and hence is an argument to the build function. This argument will be a string or an object, depending on the activation function type.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, activation_name):
        super(NeuralNetwork, self).__init__()
        self.activation = activation_functions[activation_name]
        self.fc1 = nn.Linear(28 * 28, 200)
        self.fc2 = nn.Linear(200, 200)
        self.fc3 = nn.Linear(200, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
def evaluate_model(model, test_loader, loss_fn):
    model.eval()  # Set the model to evaluation mode
    test_loss = 0
    correct = 0
    with torch.no_grad():  # No gradient computation
        for data, target in test_loader:
            output = model(data)
            y_pred = output.squeeze()
            test_loss += loss_fn(y_pred, target).item()  # Sum up batch loss
            predicted = torch.argmax(y_pred, dim=1)
            targets = torch.argmax(target, dim=1)
            correct += (predicted == targets).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = correct / len(test_loader.dataset)
    return test_loss, test_accuracy

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train_x = train_x.to(device)
train_y = train_y.to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_y = test_y.to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

In [ ]:
epochs = 5

### ReLU

ReLU or Rectified Linear Unit is simply the max function.
\begin{equation}
g(z) = \max(z, 0)
\end{equation}

In [ ]:
fig = plt.figure()
ax = plt.axes()

x = np.linspace(-10, 10, 1000)
y = np.maximum(x, 0.0)
arrowed_axes(ax)
plt.ylim(y[0]-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('relu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### Sigmoid

The sigmoid function has the form
\begin{equation}
g(z)=\sigma(z) = \frac{1}{1+e^{-z}}
\end{equation}

In [ ]:
fig = plt.figure()
ax = plt.axes()

x = np.linspace(-10, 10, 1000)
y = 1 / (1 + np.exp(-x))
arrowed_axes(ax)
plt.ylim(y[0]-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('sigmoid').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### Tanh

Tanh is the hyperbolic tangent function
\begin{equation}
g(z)=\frac{\left(e^{z}-e^{-z}\right)}{\left(e^{z}+e^{-z}\right)}
\end{equation}

In [ ]:
fig = plt.figure()
ax = plt.axes()

x = np.linspace(-10, 10, 1000)
y = np.tanh(x)
arrowed_axes(ax)
plt.ylim(y[0]-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('tanh').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### ELU

The exponential linear unit is a ReLU-like function, which can be made once differentiable if the parameter $\alpha = 1$. The functional form is given by
\begin{equation}
g(\alpha, z)=\left\{\begin{array}{ll}
			\alpha\left(e^{z}-1\right) & \text { for } z \leq 0 \\
			z & \text { for } z>0
			\end{array}\right.
\end{equation}

In [ ]:
fig = plt.figure()
ax = plt.axes()

x = np.linspace(-10, 10, 1000)
y = np.where(x > 0, x, (np.exp(x)-1))  
arrowed_axes(ax)
plt.ylim(np.min(y)-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('elu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### LeakyReLU

Leaky Rectified Linear Unit is a modified ReLU function thatprovides a non-zero output and non-zero gradient for $z < 0$. This makes training easier as the gradient is non-zero everywhere. Like ReLU is has zero second order derivatives.
\begin{equation}
g(z)=\left\{\begin{array}{ll}
			0.01 z & \text { for } z<0 \\
			z & \text { for } z \geq 0
			\end{array}\right.
\end{equation}

In [ ]:
fig = plt.figure()
ax = plt.axes()

x = np.linspace(-10, 10, 1000)
y = np.where(x > 0, x, x * 0.05)  
arrowed_axes(ax)
plt.ylim(y[0]-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

Leaky ReLU is parameterised on the alpha parameter and so this activation function is passed as an object to the model. The standard value of the parameter for leakyReLU is 0.01.

In [ ]:
model = NeuralNetwork('leaky_relu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### Swish

Swish is an activation function defined by
\begin{equation}
g(z) = z \sigma(z).
\end{equation}
Hence it is a $C^\infty$ function that is ReLU-like.

In [ ]:
fig = plt.figure()
ax = plt.axes()

x = np.linspace(-5, 5, 1000)
y = x / ( 1 + np.exp(-x))
arrowed_axes(ax)
plt.ylim(np.min(y)-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('swish').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### PReLU

The Parametric Rectified Linear Unit is a generalization of Leaky ReLU with the output for $z<0$ given by $g(z) = \alpha_i z$. PReLU treats this parameter as learnable with each layer learning a parameter value, while Leaky ReLU fixes it at the value 0.01.

In [ ]:
model = NeuralNetwork('prelu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### SELU

The scaled exponential unit was introduced as part of \textit{Self Normalizing Networks}. SELU is a variant of ELU with different parameter values selected on the basis of the Banach fixed point theorem to satisfy the self normalizing property, that is activations have mean zero and variance one.
\begin{equation}
g(\alpha, z)=\left\{\begin{array}{ll}
			\lambda\alpha\left(e^{z}-1\right) & \text { for } z \leq 0 \\
			\lambda z & \text { for } z>0
			\end{array}\right.
\end{equation}

In [ ]:
fig = plt.figure()
ax = plt.axes()

lmbda = 1.0507
alpha = 1.67326

x = np.linspace(-10, 10, 1000)
y = np.where(x > 0, lmbda * x, lmbda*(alpha * np.exp(x)-1))  
arrowed_axes(ax)
plt.ylim(np.min(y)-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('selu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### GELU

The Gaussian Error Linear Unit is based on the Gaussian cumulative distribution function. This produces a function with has similar shape to ReLU but is infinitely differentiable.

\begin{equation}
g(z) = z \Phi(z)
\end{equation}


where $\Phi(z)$ is the cumulative distribution function for $\mathcal{N}(0, 1)$. 

In [ ]:
from scipy.special import erf

fig = plt.figure()
ax = plt.axes()

x = np.linspace(-2, 2, 1000)
y = x * (1 + erf(x / np.sqrt(2))) / 2 
arrowed_axes(ax)
plt.ylim(np.min(y)-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('gelu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)

### SERLU

Scaled exponentially-regularized linear units (SERLU) are also associated with self normalizing networks.
\begin{equation}
g(z)=\lambda_{\text {serlu}}\left\{\begin{array}{cc}
				z & z \geq 0 \\
				\alpha_{\text {serlu}} z e^{z} & \text { otherwise }
			\end{array}\right.
\end{equation}
where
the optimal parameter choices are:
\begin{align}
\lambda_{\text {serlu}} = & 1.07862\\
\alpha_{\text {serlu}} = & 2.90427
\end{align}

In [ ]:
fig = plt.figure()
ax = plt.axes()

lmbda = 1.07862
alpha = 2.90427

x = np.linspace(-10, 10, 1000)
y = np.where(x > 0, lmbda * x, lmbda*(alpha * x * np.exp(x)))  
arrowed_axes(ax)
plt.ylim(np.min(y)-0.1, y[-1]+0.1)
plt.xlim(x[0], x[-1])
ax.plot(x, y, color="r", linewidth=3.0)

In [ ]:
model = NeuralNetwork('serlu').to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
t.start()
history = history_dict = train_model(model, train_loader, test_loader,
                                        loss_fn, optimizer, epochs)
t.stop()

In [ ]:
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print('test_acc:', test_acc)